In [1]:
# Assignment 9 - Q1
#
# Question:
# Implement the stack ADT twice and compare the two implementations.
#
# a) Once over a fixed-size array, with overflow and underflow detection.
# b) Once over a linked list, exposing exactly the same public interface.
# c) Test both with the same sequence of operations, confirm identical
#    results, and state one advantage of each implementation.
#
# Key Explanation:
# A stack follows LIFO (Last In, First Out).
# push() adds an element at the top.
# pop() removes and returns the top element.
# peek() returns the top element without removing it.
# is_empty() checks whether the stack is empty.


# --------------------------------------------------
# Stack using a fixed-size array
# --------------------------------------------------

class ArrayStack:

    def __init__(self, size):
        self.size = size
        self.items = [None] * size
        self.top = -1

    # Push an element
    def push(self, value):

        if self.top == self.size - 1:
            raise OverflowError("Stack overflow")

        self.top += 1
        self.items[self.top] = value

    # Pop the top element
    def pop(self):

        if self.is_empty():
            raise IndexError("Stack underflow")

        value = self.items[self.top]
        self.items[self.top] = None
        self.top -= 1

        return value

    # Return the top element
    def peek(self):

        if self.is_empty():
            raise IndexError("Stack is empty")

        return self.items[self.top]

    # Check whether stack is empty
    def is_empty(self):

        return self.top == -1


# --------------------------------------------------
# Stack using a linked list
# --------------------------------------------------

class StackNode:

    def __init__(self, data):
        self.data = data
        self.next = None


class LinkedListStack:

    def __init__(self):
        self.top = None

    # Push an element
    def push(self, value):

        node = StackNode(value)
        node.next = self.top
        self.top = node

    # Pop the top element
    def pop(self):

        if self.is_empty():
            raise IndexError("Stack underflow")

        value = self.top.data
        self.top = self.top.next

        return value

    # Return the top element
    def peek(self):

        if self.is_empty():
            raise IndexError("Stack is empty")

        return self.top.data

    # Check whether stack is empty
    def is_empty(self):

        return self.top is None


# --------------------------------------------------
# Test both stacks with the same operations
# --------------------------------------------------

array_stack = ArrayStack(5)
linked_stack = LinkedListStack()

operations = [10, 20, 30, 40]

array_results = []
linked_results = []


# Push the same elements into both stacks
for value in operations:

    array_stack.push(value)
    linked_stack.push(value)


# Check peek
array_results.append(array_stack.peek())
linked_results.append(linked_stack.peek())


# Pop elements from both stacks
while not array_stack.is_empty():

    array_results.append(array_stack.pop())
    linked_results.append(linked_stack.pop())


# Confirm identical results
print("Array stack results:", array_results)
print("Linked list stack results:", linked_results)

print("Results are identical:",
      array_results == linked_results)


# --------------------------------------------------
# Test underflow detection
# --------------------------------------------------

try:
    array_stack.pop()
except IndexError as e:
    print("Array stack:", e)


try:
    linked_stack.pop()
except IndexError as e:
    print("Linked list stack:", e)


# --------------------------------------------------
# Test overflow detection for fixed-size array
# --------------------------------------------------

small_stack = ArrayStack(3)

try:
    small_stack.push(1)
    small_stack.push(2)
    small_stack.push(3)
    small_stack.push(4)
except OverflowError as e:
    print("Array stack:", e)


# --------------------------------------------------
# Advantages
# --------------------------------------------------

# Array Stack Advantage:
# Simple and fast because elements are stored in a fixed-size array.
#
# Linked List Stack Advantage:
# It can grow dynamically and does not require a fixed capacity.

Array stack results: [40, 40, 30, 20, 10]
Linked list stack results: [40, 40, 30, 20, 10]
Results are identical: True
Array stack: Stack underflow
Linked list stack: Stack underflow
Array stack: Stack overflow


In [2]:
# Assignment 9 - Q2
#
# Question:
# Check whether an expression has balanced brackets of three kinds,
# reporting the position of the first mismatch rather than only
# True or False.
# Test at least six expressions, including correctly nested,
# wrongly nested and unclosed cases.
#
# Key Explanation:
# A stack stores opening brackets.
# When a closing bracket is found, it must match the most recent
# opening bracket.
# The position is reported using the index of the character.


def check_balanced_brackets(expression):

    stack = []

    opening = "([{"
    closing = ")]}"

    matching = {
        ')': '(',
        ']': '[',
        '}': '{'
    }

    for position, character in enumerate(expression):

        # Store opening brackets
        if character in opening:
            stack.append((character, position))

        # Check closing brackets
        elif character in closing:

            # Closing bracket without an opening bracket
            if not stack:
                return False, position

            top_bracket, top_position = stack.pop()

            # Check whether brackets match
            if top_bracket != matching[character]:
                return False, position

    # Check for unclosed opening brackets
    if stack:
        return False, stack[-1][1]

    return True, -1


# --------------------------------------------------
# Test expressions
# --------------------------------------------------

expressions = [
    "(a + b)",
    "[a + (b * c)]",
    "{[()]}",
    "([)]",
    "((a + b)",
    "{a + b]",
    "a + b"
]


# --------------------------------------------------
# Test all expressions
# --------------------------------------------------

for expression in expressions:

    result, position = check_balanced_brackets(expression)

    if result:
        print(f"{expression} -> Balanced")

    else:
        print(
            f"{expression} -> Mismatch at position {position}"
        )

(a + b) -> Balanced
[a + (b * c)] -> Balanced
{[()]} -> Balanced
([)] -> Mismatch at position 2
((a + b) -> Mismatch at position 0
{a + b] -> Mismatch at position 6
a + b -> Balanced


In [3]:
# Assignment 9 - Q3
#
# Question:
# Build an expression evaluator using stacks.
#
# a) Convert an infix expression to postfix, supporting + - * /
#    with correct precedence and brackets.
# b) Evaluate the resulting postfix expression.
# c) Chain the two so the user enters infix and receives a
#    numerical answer.
# Test at least six expressions, including nested brackets and
# one malformed input that must be rejected gracefully.
#
# Key Explanation:
# Infix expressions place operators between operands.
# Postfix expressions place operators after their operands.
# A stack is used to handle operators and operands according
# to their precedence.


# --------------------------------------------------
# Check whether a character is an operator
# --------------------------------------------------

def is_operator(character):

    return character in "+-*/"


# --------------------------------------------------
# Return operator precedence
# --------------------------------------------------

def precedence(operator):

    if operator in "+-":
        return 1

    if operator in "*/":
        return 2

    return 0


# --------------------------------------------------
# Convert infix expression to postfix
# --------------------------------------------------

def infix_to_postfix(expression):

    operators = []
    postfix = []

    i = 0

    while i < len(expression):

        character = expression[i]

        # Ignore spaces
        if character.isspace():
            i += 1
            continue

        # Read numbers
        if character.isdigit() or character == ".":

            number = ""

            while i < len(expression) and (
                expression[i].isdigit() or expression[i] == "."
            ):
                number += expression[i]
                i += 1

            # Check for invalid number
            if number.count(".") > 1:
                raise ValueError("Invalid number")

            postfix.append(number)
            continue

        # Opening bracket
        if character == "(":
            operators.append(character)

        # Closing bracket
        elif character == ")":

            found_opening = False

            while operators:

                top = operators.pop()

                if top == "(":
                    found_opening = True
                    break

                postfix.append(top)

            if not found_opening:
                raise ValueError("Unmatched closing bracket")

        # Operator
        elif is_operator(character):

            while (
                operators
                and operators[-1] != "("
                and precedence(operators[-1])
                >= precedence(character)
            ):
                postfix.append(operators.pop())

            operators.append(character)

        else:
            raise ValueError(
                f"Invalid character '{character}'"
            )

        i += 1

    # Remove remaining operators
    while operators:

        top = operators.pop()

        if top == "(":
            raise ValueError("Unclosed opening bracket")

        postfix.append(top)

    return postfix


# --------------------------------------------------
# Evaluate postfix expression
# --------------------------------------------------

def evaluate_postfix(postfix):

    stack = []

    for token in postfix:

        # Operand
        if not is_operator(token):

            stack.append(float(token))

        # Operator
        else:

            if len(stack) < 2:
                raise ValueError("Malformed expression")

            right = stack.pop()
            left = stack.pop()

            if token == "+":
                result = left + right

            elif token == "-":
                result = left - right

            elif token == "*":
                result = left * right

            elif token == "/":

                if right == 0:
                    raise ValueError("Division by zero")

                result = left / right

            stack.append(result)

    # Exactly one result should remain
    if len(stack) != 1:
        raise ValueError("Malformed expression")

    return stack[0]


# --------------------------------------------------
# Complete expression evaluator
# --------------------------------------------------

def evaluate_expression(expression):

    postfix = infix_to_postfix(expression)

    answer = evaluate_postfix(postfix)

    return postfix, answer


# --------------------------------------------------
# Test expressions
# --------------------------------------------------

expressions = [
    "2 + 3",
    "10 - 4",
    "5 * 6",
    "20 / 4",
    "2 + 3 * 4",
    "(2 + 3) * 4",
    "10 + (5 * (3 + 2))",
    "2 + * 5"
]


# --------------------------------------------------
# Test all expressions
# --------------------------------------------------

for expression in expressions:

    try:

        postfix, answer = evaluate_expression(expression)

        print("Infix:", expression)
        print("Postfix:", " ".join(postfix))
        print("Answer:", answer)
        print()

    except ValueError as error:

        print("Infix:", expression)
        print("Error:", error)
        print()


# --------------------------------------------------
# User input
# --------------------------------------------------

user_expression = input("Enter an infix expression: ")

try:

    postfix, answer = evaluate_expression(user_expression)

    print("Postfix:", " ".join(postfix))
    print("Answer:", answer)

except ValueError as error:

    print("Invalid expression:", error)

Infix: 2 + 3
Postfix: 2 3 +
Answer: 5.0

Infix: 10 - 4
Postfix: 10 4 -
Answer: 6.0

Infix: 5 * 6
Postfix: 5 6 *
Answer: 30.0

Infix: 20 / 4
Postfix: 20 4 /
Answer: 5.0

Infix: 2 + 3 * 4
Postfix: 2 3 4 * +
Answer: 14.0

Infix: (2 + 3) * 4
Postfix: 2 3 + 4 *
Answer: 20.0

Infix: 10 + (5 * (3 + 2))
Postfix: 10 5 3 2 + * +
Answer: 35.0

Infix: 2 + * 5
Error: Malformed expression



Enter an infix expression:  (2 + 3) * 4


Postfix: 2 3 + 4 *
Answer: 20.0


In [4]:
# Assignment 9 - Q4
#
# Question:
# Demonstrate why the circular queue exists.
# a) Implement a simple array queue, then fill it, empty it,
#    and show that it still reports itself full.
# b) Implement a circular queue that fixes the problem.
# c) State in a comment which full-versus-empty convention
#    you chose, and why.
#
# Key Explanation:
# A simple array queue may leave unused spaces at the beginning
# after dequeue operations, but a rear pointer may still reach the end.
# A circular queue reuses these empty spaces by wrapping around.
#
# Full-versus-empty convention:
# We use size to track the number of elements.
# size == 0 means empty and size == capacity means full.


# --------------------------------------------------
# Part A: Simple Array Queue
# --------------------------------------------------

class SimpleQueue:

    def __init__(self, capacity):
        self.capacity = capacity
        self.queue = [None] * capacity
        self.front = 0
        self.rear = -1
        self.size = 0

    # Add an element
    def enqueue(self, value):

        if self.rear == self.capacity - 1:
            raise OverflowError("Queue is full")

        self.rear += 1
        self.queue[self.rear] = value
        self.size += 1

    # Remove an element
    def dequeue(self):

        if self.size == 0:
            raise IndexError("Queue is empty")

        value = self.queue[self.front]
        self.queue[self.front] = None
        self.front += 1
        self.size -= 1

        return value

    # Check whether queue is full
    def is_full(self):

        return self.rear == self.capacity - 1


# Fill the simple queue
simple_queue = SimpleQueue(5)

for value in [10, 20, 30, 40, 50]:
    simple_queue.enqueue(value)

print("Simple Queue is full:", simple_queue.is_full())


# Empty the queue
while simple_queue.size > 0:
    simple_queue.dequeue()

print("Simple Queue after emptying:")
print("Front:", simple_queue.front)
print("Rear:", simple_queue.rear)
print("Is full:", simple_queue.is_full())


# --------------------------------------------------
# Part B: Circular Queue
# --------------------------------------------------

class CircularQueue:

    def __init__(self, capacity):
        self.capacity = capacity
        self.queue = [None] * capacity
        self.front = 0
        self.rear = 0
        self.size = 0

    # Add an element
    def enqueue(self, value):

        if self.size == self.capacity:
            raise OverflowError("Queue is full")

        self.queue[self.rear] = value

        self.rear = (self.rear + 1) % self.capacity

        self.size += 1

    # Remove an element
    def dequeue(self):

        if self.size == 0:
            raise IndexError("Queue is empty")

        value = self.queue[self.front]

        self.queue[self.front] = None

        self.front = (self.front + 1) % self.capacity

        self.size -= 1

        return value

    # Check whether queue is full
    def is_full(self):

        return self.size == self.capacity

    # Check whether queue is empty
    def is_empty(self):

        return self.size == 0


# Create circular queue
circular_queue = CircularQueue(5)


# Fill the circular queue
for value in [10, 20, 30, 40, 50]:
    circular_queue.enqueue(value)

print("\nCircular Queue is full:", circular_queue.is_full())


# Remove some elements
circular_queue.dequeue()
circular_queue.dequeue()


# Add new elements into the freed spaces
circular_queue.enqueue(60)
circular_queue.enqueue(70)


print("Circular Queue after reusing freed spaces:")
print("Queue:", circular_queue.queue)
print("Is full:", circular_queue.is_full())


# The circular queue reuses the spaces released by dequeue.
#
# Full condition:
# size == capacity
#
# Empty condition:
# size == 0
#
# This convention is used because size clearly distinguishes
# a full queue from an empty queue.

Simple Queue is full: True
Simple Queue after emptying:
Front: 5
Rear: 4
Is full: True

Circular Queue is full: True
Circular Queue after reusing freed spaces:
Queue: [60, 70, 30, 40, 50]
Is full: True


In [5]:
# Assignment 9 - Q4
#
# Question:
# Demonstrate why the circular queue exists.
#
# a) Implement a simple array queue, then fill it, empty it,
#    and show that it still reports itself full.
#
# b) Implement a circular queue that fixes the problem.
#
# c) State in a comment which full-versus-empty convention
#    you chose, and why.
#
# Key Explanation:
# In a simple array queue, after deleting elements, empty spaces
# at the beginning cannot be reused when rear has reached the end.
# A circular queue solves this problem by wrapping rear back to
# the beginning of the array.
#
# Convention used:
# size == 0  -> queue is empty
# size == capacity -> queue is full
#
# The size variable is used because it clearly distinguishes
# between full and empty states.


# --------------------------------------------------
# Part A: Simple Array Queue
# --------------------------------------------------

class SimpleQueue:

    def __init__(self, capacity):
        self.capacity = capacity
        self.queue = [None] * capacity
        self.front = 0
        self.rear = -1
        self.size = 0

    def enqueue(self, value):

        if self.rear == self.capacity - 1:
            raise OverflowError("Queue is full")

        self.rear += 1
        self.queue[self.rear] = value
        self.size += 1

    def dequeue(self):

        if self.size == 0:
            raise IndexError("Queue is empty")

        value = self.queue[self.front]
        self.queue[self.front] = None
        self.front += 1
        self.size -= 1

        return value

    def is_full(self):

        return self.rear == self.capacity - 1


# Create simple queue
simple_queue = SimpleQueue(5)


# Fill the queue
for value in [10, 20, 30, 40, 50]:
    simple_queue.enqueue(value)


# Empty the queue
while simple_queue.size > 0:
    simple_queue.dequeue()


# The queue is empty, but rear is still at the last index.
print("Simple queue is full:", simple_queue.is_full())


# --------------------------------------------------
# Part B: Circular Queue
# --------------------------------------------------

class CircularQueue:

    def __init__(self, capacity):

        self.capacity = capacity
        self.queue = [None] * capacity
        self.front = 0
        self.rear = 0
        self.size = 0

    def enqueue(self, value):

        if self.size == self.capacity:
            raise OverflowError("Queue is full")

        self.queue[self.rear] = value

        # Move rear circularly
        self.rear = (self.rear + 1) % self.capacity

        self.size += 1

    def dequeue(self):

        if self.size == 0:
            raise IndexError("Queue is empty")

        value = self.queue[self.front]
        self.queue[self.front] = None

        # Move front circularly
        self.front = (self.front + 1) % self.capacity

        self.size -= 1

        return value

    def is_full(self):

        return self.size == self.capacity

    def is_empty(self):

        return self.size == 0


# Create circular queue
circular_queue = CircularQueue(5)


# Fill the queue
for value in [10, 20, 30, 40, 50]:
    circular_queue.enqueue(value)


# Delete two elements
circular_queue.dequeue()
circular_queue.dequeue()


# Reuse the freed spaces
circular_queue.enqueue(60)
circular_queue.enqueue(70)


print("Circular queue:", circular_queue.queue)
print("Circular queue is full:", circular_queue.is_full())


# --------------------------------------------------
# Convention:
# --------------------------------------------------
#
# Empty condition:
# size == 0
#
# Full condition:
# size == capacity
#
# This convention is chosen because the size variable
# clearly distinguishes an empty queue from a full queue.
#
# Circular movement:
# rear = (rear + 1) % capacity
# front = (front + 1) % capacity
#
# This allows the queue to reuse empty positions at
# the beginning of the array.

Simple queue is full: True
Circular queue: [60, 70, 30, 40, 50]
Circular queue is full: True


In [6]:
# Assignment 9 - Q5
#
# Question:
# Use your circular queue to simulate round-robin scheduling of
# four processes with a fixed time quantum.
# Print the order in which the processes run and the completion
# time of each.
# Repeat with quanta of 2 and 4, and state which gave the lower
# average completion time.
#
# Key Explanation:
# Round-robin scheduling gives each process a fixed time quantum.
# If a process is not finished after its quantum, it is placed
# at the rear of the circular queue.
# A process's completion time is the time when its final burst ends.
#
# Four processes are used with CPU burst times:
# P1 = 4, P2 = 2, P3 = 3, P4 = 5.
#
# The same processes are tested with time quantum 2 and 4.


class CircularQueue:

    def __init__(self, capacity):
        self.capacity = capacity
        self.data = [None] * capacity
        self.front = 0
        self.count = 0

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.capacity

    def enqueue(self, value):

        if self.is_full():
            raise Exception("Queue overflow")

        rear = (self.front + self.count) % self.capacity
        self.data[rear] = value
        self.count += 1

    def dequeue(self):

        if self.is_empty():
            raise Exception("Queue underflow")

        value = self.data[self.front]
        self.data[self.front] = None
        self.front = (self.front + 1) % self.capacity
        self.count -= 1

        return value


def round_robin(processes, quantum):

    # Create circular queue
    queue = CircularQueue(len(processes))

    # Store remaining burst time
    remaining = {}

    # Store completion time
    completion = {}

    # Add all processes to the queue
    for process, burst in processes.items():
        remaining[process] = burst
        queue.enqueue(process)

    current_time = 0
    run_order = []

    # Continue until all processes are completed
    while not queue.is_empty():

        process = queue.dequeue()

        # Process runs for quantum or until it finishes
        run_time = min(quantum, remaining[process])

        run_order.append(process)

        current_time += run_time
        remaining[process] -= run_time

        # If process is not finished, put it back
        if remaining[process] > 0:
            queue.enqueue(process)

        else:
            completion[process] = current_time

    return run_order, completion


# --------------------------------------------------
# Four processes
# --------------------------------------------------

processes = {
    "P1": 4,
    "P2": 2,
    "P3": 3,
    "P4": 5
}


# --------------------------------------------------
# Test with quantum = 2
# --------------------------------------------------

run_order_2, completion_2 = round_robin(processes, 2)

average_2 = sum(completion_2.values()) / len(completion_2)

print("Time Quantum = 2")
print("Run order:", " -> ".join(run_order_2))
print("Completion times:", completion_2)
print("Average completion time:", average_2)


# --------------------------------------------------
# Test with quantum = 4
# --------------------------------------------------

run_order_4, completion_4 = round_robin(processes, 4)

average_4 = sum(completion_4.values()) / len(completion_4)

print("\nTime Quantum = 4")
print("Run order:", " -> ".join(run_order_4))
print("Completion times:", completion_4)
print("Average completion time:", average_4)


# --------------------------------------------------
# Compare average completion times
# --------------------------------------------------

if average_2 < average_4:
    print("\nQuantum 2 gives the lower average completion time.")

elif average_4 < average_2:
    print("\nQuantum 4 gives the lower average completion time.")

else:
    print("\nBoth quanta give the same average completion time.")


# Key Explanation:
# With quantum 2, processes receive shorter CPU turns and are
# returned to the circular queue more frequently.
#
# With quantum 4, processes can complete more work in each turn.
#
# The average completion time is calculated as:
#
# Average completion time =
# Sum of completion times / Number of processes

Time Quantum = 2
Run order: P1 -> P2 -> P3 -> P4 -> P1 -> P3 -> P4 -> P4
Completion times: {'P2': 4, 'P1': 10, 'P3': 11, 'P4': 14}
Average completion time: 9.75

Time Quantum = 4
Run order: P1 -> P2 -> P3 -> P4 -> P4
Completion times: {'P1': 4, 'P2': 6, 'P3': 9, 'P4': 14}
Average completion time: 8.25

Quantum 4 gives the lower average completion time.


In [7]:
# ============================================================
# ASSIGNMENT 10 - FINAL PROJECT
# TICKET BOOKING QUEUE
# ============================================================
#
# Question / Project Requirement:
# Simulate a ticket counter where customers arrive and are served
# in arrival order. Allow cancellations and compare a circular
# queue with a linked-list queue, considering that cancellation
# may remove someone from the middle.
#
# Key Explanation:
# A queue follows FIFO (First In, First Out).
# The customer who arrives first is served first.
#
# Circular Queue:
# Uses a fixed-size array and reuses empty positions by wrapping
# around from the end to the beginning.
#
# Linked-List Queue:
# Uses nodes connected with references. It can grow dynamically.
#
# Middle cancellation:
# If a customer cancels before being served, that customer must be
# removed from the middle while maintaining the correct queue order.
#
# ============================================================


# ============================================================
# PART 1 - CIRCULAR QUEUE
# ============================================================

class CircularTicketQueue:

    def __init__(self, capacity):

        self.capacity = capacity
        self.queue = [None] * capacity

        self.front = 0
        self.rear = 0
        self.size = 0

    # --------------------------------------------------------
    # Check whether queue is empty
    # --------------------------------------------------------

    def is_empty(self):

        return self.size == 0

    # --------------------------------------------------------
    # Check whether queue is full
    # --------------------------------------------------------

    def is_full(self):

        return self.size == self.capacity

    # --------------------------------------------------------
    # Add customer to queue
    # --------------------------------------------------------

    def enqueue(self, customer):

        if self.is_full():
            return False

        self.queue[self.rear] = customer

        self.rear = (self.rear + 1) % self.capacity

        self.size += 1

        return True

    # --------------------------------------------------------
    # Serve the first customer
    # --------------------------------------------------------

    def dequeue(self):

        if self.is_empty():
            return None

        customer = self.queue[self.front]

        self.queue[self.front] = None

        self.front = (self.front + 1) % self.capacity

        self.size -= 1

        return customer

    # --------------------------------------------------------
    # Cancel a customer
    # --------------------------------------------------------

    def cancel(self, customer_id):

        if self.is_empty():
            return False

        customers = []

        current = self.front

        for _ in range(self.size):

            customers.append(self.queue[current])

            current = (current + 1) % self.capacity

        found = False

        for customer in customers:

            if customer["id"] == customer_id:

                found = True

                break

        if not found:
            return False

        customers = [
            customer
            for customer in customers
            if customer["id"] != customer_id
        ]

        self.queue = [None] * self.capacity

        self.front = 0
        self.rear = 0
        self.size = 0

        for customer in customers:
            self.enqueue(customer)

        return True

    # --------------------------------------------------------
    # Display queue
    # --------------------------------------------------------

    def display(self):

        if self.is_empty():

            print("Queue is empty.")

            return

        print("\nCircular Queue:")

        current = self.front

        for position in range(1, self.size + 1):

            customer = self.queue[current]

            print(
                position,
                "->",
                customer["id"],
                customer["name"]
            )

            current = (current + 1) % self.capacity


# ============================================================
# PART 2 - LINKED LIST QUEUE
# ============================================================

class TicketNode:

    def __init__(self, customer):

        self.customer = customer
        self.next = None


class LinkedListTicketQueue:

    def __init__(self):

        self.front = None
        self.rear = None

    # --------------------------------------------------------
    # Check whether queue is empty
    # --------------------------------------------------------

    def is_empty(self):

        return self.front is None

    # --------------------------------------------------------
    # Add customer to queue
    # --------------------------------------------------------

    def enqueue(self, customer):

        new_node = TicketNode(customer)

        if self.front is None:

            self.front = new_node
            self.rear = new_node

            return True

        self.rear.next = new_node

        self.rear = new_node

        return True

    # --------------------------------------------------------
    # Serve first customer
    # --------------------------------------------------------

    def dequeue(self):

        if self.is_empty():

            return None

        customer = self.front.customer

        self.front = self.front.next

        if self.front is None:

            self.rear = None

        return customer

    # --------------------------------------------------------
    # Cancel a customer from any position
    # --------------------------------------------------------

    def cancel(self, customer_id):

        if self.is_empty():

            return False

        current = self.front
        previous = None

        while current is not None:

            if current.customer["id"] == customer_id:

                # Deleting the first customer
                if previous is None:

                    self.front = current.next

                # Deleting a middle or last customer
                else:

                    previous.next = current.next

                # If the deleted customer was the rear
                if current == self.rear:

                    self.rear = previous

                return True

            previous = current

            current = current.next

        return False

    # --------------------------------------------------------
    # Display queue
    # --------------------------------------------------------

    def display(self):

        if self.is_empty():

            print("Queue is empty.")

            return

        print("\nLinked List Queue:")

        current = self.front

        position = 1

        while current is not None:

            customer = current.customer

            print(
                position,
                "->",
                customer["id"],
                customer["name"]
            )

            current = current.next

            position += 1


# ============================================================
# PART 3 - CUSTOMER CREATION
# ============================================================

def create_customer(customer_id, name):

    return {
        "id": customer_id,
        "name": name
    }


customers = [

    create_customer(101, "Aarav"),
    create_customer(102, "Priya"),
    create_customer(103, "Rahul"),
    create_customer(104, "Sneha"),
    create_customer(105, "Rohan"),
    create_customer(106, "Ananya"),
    create_customer(107, "Vikas"),
    create_customer(108, "Neha")
]


# ============================================================
# PART 4 - CREATE BOTH QUEUES
# ============================================================

circular_queue = CircularTicketQueue(20)

linked_queue = LinkedListTicketQueue()


# ============================================================
# PART 5 - ADD CUSTOMERS
# ============================================================

for customer in customers:

    circular_queue.enqueue(customer)

    linked_queue.enqueue(customer)


print("INITIAL QUEUES")

circular_queue.display()

linked_queue.display()


# ============================================================
# PART 6 - CANCEL CUSTOMER FROM MIDDLE
# ============================================================

customer_to_cancel = 104


circular_cancelled = circular_queue.cancel(
    customer_to_cancel
)

linked_cancelled = linked_queue.cancel(
    customer_to_cancel
)


print("\nCANCELLATION")

print(
    "Circular Queue cancellation:",
    circular_cancelled
)

print(
    "Linked List Queue cancellation:",
    linked_cancelled
)


# ============================================================
# PART 7 - DISPLAY QUEUES AFTER CANCELLATION
# ============================================================

print("\nQUEUES AFTER MIDDLE CANCELLATION")

circular_queue.display()

linked_queue.display()


# ============================================================
# PART 8 - SERVE CUSTOMERS IN FIFO ORDER
# ============================================================

print("\nSERVING CUSTOMERS - CIRCULAR QUEUE")

while not circular_queue.is_empty():

    customer = circular_queue.dequeue()

    print(
        "Served:",
        customer["id"],
        customer["name"]
    )


print("\nSERVING CUSTOMERS - LINKED LIST QUEUE")

while not linked_queue.is_empty():

    customer = linked_queue.dequeue()

    print(
        "Served:",
        customer["id"],
        customer["name"]
    )


# ============================================================
# PART 9 - TESTING WITH 10,000 CUSTOMERS
# ============================================================
#
# Key Explanation:
# The workbook requires performance measurement using at least
# 10,000 records or an equivalent workload.
#
# Here, 10,000 customer records are created for testing.
# ============================================================


import time


large_customers = [

    create_customer(
        i,
        "Customer_" + str(i)
    )

    for i in range(1, 10001)
]


# ============================================================
# PART 10 - PERFORMANCE TEST:
# CIRCULAR QUEUE
# ============================================================

large_circular_queue = CircularTicketQueue(10000)


start_time = time.perf_counter()


for customer in large_customers:

    large_circular_queue.enqueue(customer)


circular_enqueue_time = (
    time.perf_counter() - start_time
)


# Cancel a middle customer
start_time = time.perf_counter()


large_circular_queue.cancel(5000)


circular_cancel_time = (
    time.perf_counter() - start_time
)


# ============================================================
# PART 11 - PERFORMANCE TEST:
# LINKED LIST QUEUE
# ============================================================

large_linked_queue = LinkedListTicketQueue()


start_time = time.perf_counter()


for customer in large_customers:

    large_linked_queue.enqueue(customer)


linked_enqueue_time = (
    time.perf_counter() - start_time
)


# Cancel a middle customer
start_time = time.perf_counter()


large_linked_queue.cancel(5000)


linked_cancel_time = (
    time.perf_counter() - start_time
)


# ============================================================
# PART 12 - PERFORMANCE MEASUREMENT
# ============================================================

print("\nPERFORMANCE MEASUREMENT")

print(
    "Number of records:",
    len(large_customers)
)

print(
    "\nCircular Queue Enqueue Time:",
    circular_enqueue_time,
    "seconds"
)

print(
    "Circular Queue Middle Cancellation Time:",
    circular_cancel_time,
    "seconds"
)

print(
    "\nLinked List Queue Enqueue Time:",
    linked_enqueue_time,
    "seconds"
)

print(
    "Linked List Queue Middle Cancellation Time:",
    linked_cancel_time,
    "seconds"
)


# ============================================================
# PART 13 - COMPARISON
# ============================================================

print("\nDATA STRUCTURE COMPARISON")

print(
    "Circular Queue:"
)

print(
    "- Fixed capacity"
)

print(
    "- Efficient FIFO enqueue and dequeue"
)

print(
    "- Uses array storage"
)

print(
    "- Middle cancellation requires rearranging/rebuilding"
)


print(
    "\nLinked List Queue:"
)

print(
    "- Dynamic size"
)

print(
    "- Efficient FIFO enqueue and dequeue"
)

print(
    "- Uses linked nodes"
)

print(
    "- Middle cancellation can be performed by changing links"
)


# ============================================================
# PART 14 - FINAL TEST
# ============================================================

test_circular = CircularTicketQueue(5)

test_linked = LinkedListTicketQueue()


test_customers = [

    create_customer(1, "Customer A"),
    create_customer(2, "Customer B"),
    create_customer(3, "Customer C"),
    create_customer(4, "Customer D"),
    create_customer(5, "Customer E")
]


for customer in test_customers:

    test_circular.enqueue(customer)

    test_linked.enqueue(customer)


# Cancel a middle customer
test_circular.cancel(3)

test_linked.cancel(3)


print("\nFINAL TEST - AFTER CANCELLING CUSTOMER 3")

test_circular.display()

test_linked.display()


# Serve the remaining customers
print("\nFINAL TEST - SERVING CUSTOMERS")

print("\nCircular Queue:")

while not test_circular.is_empty():

    customer = test_circular.dequeue()

    print(
        customer["id"],
        customer["name"]
    )


print("\nLinked List Queue:")

while not test_linked.is_empty():

    customer = test_linked.dequeue()

    print(
        customer["id"],
        customer["name"]
    )


# ============================================================
# KEY COMPLEXITY NOTES
# ============================================================
#
# Circular Queue:
# Enqueue -> O(1)
# Dequeue -> O(1)
# Middle cancellation -> O(n)
#
# Linked List Queue:
# Enqueue -> O(1) when rear reference is maintained
# Dequeue -> O(1)
# Middle cancellation -> O(n) to find the customer,
# but the actual link removal is O(1) once the node is found.
#
# Main comparison:
# The circular queue provides efficient array-based FIFO operations.
# The linked-list queue provides dynamic storage and easier node
# removal without shifting array elements.
#
# The project demonstrates how the choice of data structure affects
# the implementation of a real-world ticket booking queue.

INITIAL QUEUES

Circular Queue:
1 -> 101 Aarav
2 -> 102 Priya
3 -> 103 Rahul
4 -> 104 Sneha
5 -> 105 Rohan
6 -> 106 Ananya
7 -> 107 Vikas
8 -> 108 Neha

Linked List Queue:
1 -> 101 Aarav
2 -> 102 Priya
3 -> 103 Rahul
4 -> 104 Sneha
5 -> 105 Rohan
6 -> 106 Ananya
7 -> 107 Vikas
8 -> 108 Neha

CANCELLATION
Circular Queue cancellation: True
Linked List Queue cancellation: True

QUEUES AFTER MIDDLE CANCELLATION

Circular Queue:
1 -> 101 Aarav
2 -> 102 Priya
3 -> 103 Rahul
4 -> 105 Rohan
5 -> 106 Ananya
6 -> 107 Vikas
7 -> 108 Neha

Linked List Queue:
1 -> 101 Aarav
2 -> 102 Priya
3 -> 103 Rahul
4 -> 105 Rohan
5 -> 106 Ananya
6 -> 107 Vikas
7 -> 108 Neha

SERVING CUSTOMERS - CIRCULAR QUEUE
Served: 101 Aarav
Served: 102 Priya
Served: 103 Rahul
Served: 105 Rohan
Served: 106 Ananya
Served: 107 Vikas
Served: 108 Neha

SERVING CUSTOMERS - LINKED LIST QUEUE
Served: 101 Aarav
Served: 102 Priya
Served: 103 Rahul
Served: 105 Rohan
Served: 106 Ananya
Served: 107 Vikas
Served: 108 Neha

PERFORMANCE ME